# 2. Baseline modeling — 3D U-Net + transformer

Thin driver around the vendored baseline in `scripts/` (see
`docs/0_coding_standards.md` for why that logic lives in `scripts/` rather
than `src/` for now). Trains/predicts with the temporal-attention U-Net +
cross-attention transformer edge predictor, then (in `submission` mode)
produces `submission.csv` for upload.

Not yet run: needs the competition data, which isn't downloaded locally —
run on Kaggle Kernels (see `../kaggle/README.md`) or point
`$CELLMOT_DATA_DIR` at a local copy.

## Config

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent

SEED = 0
RUN_MODE = "train"  # "train" | "submission"
METHOD = "baseline"
SPLIT = "0"
EPOCHS = 3


def run(*args: str) -> None:
    """Run a vendored script with the current kernel's interpreter."""
    subprocess.run([sys.executable, *args], check=True, cwd=REPO_ROOT)

## Train

In [ ]:
if RUN_MODE == "train":
    run(
        "scripts/train_unet_transformer.py",
        "--split", SPLIT,
        "--epochs", str(EPOCHS),
    )

*Insight: fill in after running — training loss curve, whether it converged
in `EPOCHS` epochs, any stability issues.*

## Predict + build submission (submission mode only)

In [ ]:
if RUN_MODE == "submission":
    run("scripts/predict_unet_transformer.py", "--method", METHOD, "--split", SPLIT)

    import os
    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    predictions_dir = REPO_ROOT / "predictions" / kaggle_user / METHOD / f"split_{SPLIT}"
    submission_csv = REPO_ROOT / "submission.csv"

    run(
        "scripts/geffs_to_csv.py",
        "--in-dir", str(predictions_dir),
        "--csv", str(submission_csv),
    )
    print(f"Wrote {submission_csv}")

## Evaluate locally (sanity check before uploading)

In [ ]:
if RUN_MODE == "submission":
    run(
        "scripts/csv_to_geffs.py",
        "--csv", str(REPO_ROOT / "submission.csv"),
        "--out-dir", str(REPO_ROOT / "out_geffs"),
    )
    run(
        "scripts/evaluate.py",
        "--pred-dir", str(REPO_ROOT / "out_geffs"),
        "--gt-dir", str(REPO_ROOT / "data" / "train"),
    )

*Insight: fill in after running — edge Jaccard, division Jaccard, final
score, and how it compares to the previous best (record in `README.md`).*

## Findings / limitations / next experiment

- **Findings**: _fill in after running._
- **Limitations**: baseline weights are trained from scratch here, not to
  convergence at low `EPOCHS` — treat early runs as a smoke test, not a
  real score.
- **Next**: sweep `--det-threshold` / try `--use-ilp` in
  `predict_unet_transformer.py`; train longer once a run completes cleanly
  on Kaggle Kernels.